# Evaluation: final scores on the test split

This notebook scores every model on the held-out test split, once. The protocol:

1. Build each model with the settings stored in `models.BEST_PARAMS`, which were chosen on the validation split in `03_models.ipynb`. The majority baseline has no settings.
2. Fit on the training split only. The validation split is not added back, so the tuned settings stay matched to the amount of data they were tuned on.
3. Predict class probabilities for the test split one time and keep them in `test_proba`. Every later section reads those probabilities. Nothing is refitted or rescored after this point, and no setting changes after the test numbers are seen.
4. Report average AUC and macro-F1 from `src/metrics.py`, each with a 95% bootstrap interval.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))

import pandas as pd  # noqa: E402

from src import data, metrics, models  # noqa: E402

In [ ]:
df = data.load()
train, valid, test = data.split(df)
train_p, valid_p, test_p = data.preprocess(train, valid, test)  # fills use train statistics only
y_test = test_p["fit"]
print(f"train {len(train_p)}, test {len(test_p)}")

## Fit on train, score test once

The next cell is the only place the test split is scored. It fits the three models and stores one probability table per model in `test_proba`.

In [ ]:
# The test split is scored here, once. Later sections reuse test_proba.
final = {"MajorityClass": models.MajorityClass()}
final.update({name: getattr(models, name)(**params) for name, params in models.BEST_PARAMS.items()})
print({name: params for name, params in models.BEST_PARAMS.items()})

test_proba = {name: model.fit(train_p).predict_proba(test_p) for name, model in final.items()}

## Results table

Each interval comes from 1000 bootstrap resamples of the test rows, drawn with a fixed seed. The models are already fitted, so the interval shows how much the score moves with the choice of test rows. It does not include the variation from refitting on a different training sample.

In [ ]:
N_BOOT, BOOT_SEED = 1000, 0

rows = {}
for name, proba in test_proba.items():
    ci = metrics.bootstrap_ci(y_test, proba, n_boot=N_BOOT, seed=BOOT_SEED)
    rows[name] = {
        "average AUC": metrics.average_auc(y_test, proba),
        "AUC 95% interval": "[{:.4f}, {:.4f}]".format(*ci["average AUC"]),
        "macro-F1": metrics.macro_f1(y_test, proba.idxmax(axis=1)),
        "macro-F1 95% interval": "[{:.4f}, {:.4f}]".format(*ci["macro-F1"]),
    }
test_results = pd.DataFrame(rows).T
test_results = test_results.astype({"average AUC": float, "macro-F1": float}).round(4)
print(test_results.to_string())
test_results

## Does the main model win?

Yes, on both metrics. On the full data the latent factor model scores an average AUC of 0.7578 (95% interval 0.7504 to 0.7647) against 0.6203 for logistic regression and 0.5000 for the majority baseline. Its macro-F1 is 0.5237 (0.5152 to 0.5309) against 0.3316 and 0.2830. The gap to the course model is 0.138 in AUC and 0.192 in macro-F1. Each interval is about 0.014 to 0.016 wide, so the gaps are roughly ten times the uncertainty and no interval overlaps another. Test scores sit close to the validation scores from `03_models.ipynb` (0.7570 and 0.5181 for the main model, 0.6188 and 0.3341 for logistic regression), so tuning on validation did not overfit it.

The interval covers the choice of test rows only. It does not cover refitting on a different training sample. The committed copy of this notebook runs on the 200-row sample in CI, so its table differs from the numbers quoted here, which come from the full data.